In [1]:
# setup
import pandas as pd
import numpy as np

df = pd.read_csv("Filemaker_CLEAN.csv")

df = df[[
    "uniqname",
    "experience_type",
    "year",
    "term_season"
]]

df = df.dropna()
df["experience_type"] = df["experience_type"].str.strip()

# Remove exact duplicates
df = df.drop_duplicates()

In [2]:
# Time ordering
season_map = {"WN": 1, "SP": 2, "SU": 3, "FA": 4}
df["season_num"] = df["term_season"].map(season_map)

df = df.sort_values(["uniqname", "year", "season_num"])

In [3]:
# Build sequences

# Raw sequences (keep repeats)
raw_sequences = df.groupby("uniqname")["experience_type"].apply(list)

# Collapse consecutive duplicates
def remove_consecutive_duplicates(seq):
    return [seq[i] for i in range(len(seq)) if i == 0 or seq[i] != seq[i-1]]

collapsed_sequences = raw_sequences.apply(remove_consecutive_duplicates)

# First Touch Analysis + Pathways

In [4]:

# RAW
first_touch_raw = raw_sequences.apply(lambda x: x[0] if len(x) > 0 else None)

print("\n=== First Contact (RAW) ===")
print(first_touch_raw.value_counts(normalize=True))


=== First Contact (RAW) ===
experience_type
Event                     0.590486
Course                    0.245708
Counseling Appointment    0.090589
Funding                   0.073216
Name: proportion, dtype: float64


In [5]:
# Collapsed
first_touch_collapsed = collapsed_sequences.apply(lambda x: x[0] if len(x) > 0 else None)

print("\n=== First Touch (COLLAPSED) ===")
print(first_touch_collapsed.value_counts(normalize=True))


=== First Touch (COLLAPSED) ===
experience_type
Event                     0.590486
Course                    0.245708
Counseling Appointment    0.090589
Funding                   0.073216
Name: proportion, dtype: float64


In [6]:
# Transitions RAW
raw_transitions = []

for seq in raw_sequences:
    for i in range(len(seq) - 1):
        raw_transitions.append((seq[i], seq[i+1]))

raw_transitions_df = pd.DataFrame(raw_transitions, columns=["current", "next"])

raw_counts = (
    raw_transitions_df.groupby(["current", "next"])
    .size()
    .reset_index(name="count")
)

raw_counts["total"] = raw_counts.groupby("current")["count"].transform("sum")
raw_counts["prob"] = raw_counts["count"] / raw_counts["total"]

# Baseline
baseline = df["experience_type"].value_counts(normalize=True)
raw_counts["baseline"] = raw_counts["next"].map(baseline)
raw_counts["lift"] = raw_counts["prob"] / raw_counts["baseline"]


print("\n=== RAW TRANSITIONS ===")
print(raw_counts.sort_values("count", ascending=False).head(20))


=== RAW TRANSITIONS ===
                   current                    next  count  total      prob  \
10                   Event                   Event   1154   2777  0.415556   
9                    Event                  Course    714   2777  0.257112   
2   Counseling Appointment                   Event    596   1181  0.504657   
5                   Course                  Course    566   1658  0.341375   
6                   Course                   Event    549   1658  0.331122   
8                    Event  Counseling Appointment    469   2777  0.168887   
11                   Event                 Funding    440   2777  0.158444   
4                   Course  Counseling Appointment    397   1658  0.239445   
14                 Funding                   Event    282    668  0.422156   
1   Counseling Appointment                  Course    252   1181  0.213378   
3   Counseling Appointment                 Funding    194   1181  0.164268   
12                 Funding  Counseling 

In [7]:
# Transitions COLLAPSED
collapsed_transitions = []

for seq in collapsed_sequences:
    for i in range(len(seq) - 1):
        collapsed_transitions.append((seq[i], seq[i+1]))

collapsed_df = pd.DataFrame(collapsed_transitions, columns=["current", "next"])

collapsed_counts = (
    collapsed_df.groupby(["current", "next"])
    .size()
    .reset_index(name="count")
)

collapsed_counts["total"] = collapsed_counts.groupby("current")["count"].transform("sum")
collapsed_counts["prob"] = collapsed_counts["count"] / collapsed_counts["total"]

collapsed_counts["baseline"] = collapsed_counts["next"].map(baseline)
collapsed_counts["lift"] = collapsed_counts["prob"] / collapsed_counts["baseline"]

collapsed_counts = collapsed_counts.sort_values("lift", ascending=False).reset_index(drop=True)

print("\n=== COLLAPSED TRANSITIONS ===")
print(collapsed_counts.sort_values("lift", ascending=False).head(20))


=== COLLAPSED TRANSITIONS ===
                   current                    next  count  total      prob  \
0                    Event                 Funding    440   1623  0.271103   
1                   Course  Counseling Appointment    397   1092  0.363553   
2                  Funding  Counseling Appointment    190    588  0.323129   
3                    Event  Counseling Appointment    469   1623  0.288971   
4                    Event                  Course    714   1623  0.439926   
5   Counseling Appointment                 Funding    194   1042  0.186180   
6                   Course                 Funding    146   1092  0.133700   
7   Counseling Appointment                   Event    596   1042  0.571977   
8                   Course                   Event    549   1092  0.502747   
9                  Funding                   Event    282    588  0.479592   
10  Counseling Appointment                  Course    252   1042  0.241843   
11                 Funding       

# High Engagement Analysis

In [8]:
# Define High Engagement
engagement_counts = df.groupby("uniqname").size().rename("total_engagements")

student_df = engagement_counts.reset_index()

threshold = student_df["total_engagements"].quantile(0.75)

student_df["high_engagement"] = (student_df["total_engagements"] >= threshold).astype(int)

In [9]:
# RAW Features
raw_flags = (
    df.assign(val=1)
    .pivot_table(index="uniqname", columns="experience_type", values="val", aggfunc="max", fill_value=0)
)

features_raw = student_df.set_index("uniqname").join(raw_flags)

In [10]:
# EARLY Features
df["order"] = df.groupby("uniqname").cumcount()

early_df = df[df["order"] <= 1]

early_flags = (
    early_df.assign(val=1)
    .pivot_table(index="uniqname", columns="experience_type", values="val", aggfunc="max", fill_value=0)
)

features_early = student_df.set_index("uniqname").join(early_flags)

In [11]:
# COLLAPSED Features
collapsed_flags = pd.DataFrame(index=collapsed_sequences.index)

for exp_type in df["experience_type"].unique():
    collapsed_flags[exp_type] = collapsed_sequences.apply(lambda seq: int(exp_type in seq))

features_collapsed = student_df.set_index("uniqname").join(collapsed_flags)

In [12]:
# General Impact Function (use on each feature set)

def compute_lift(features, label=""):
    results = {}

    for col in features.columns:
        if col in ["total_engagements", "high_engagement"]:
            continue
        
        group = features.groupby(col)["high_engagement"].mean()
        
        if 0 in group and 1 in group:
            results[col] = {
                "No": group[0],
                "Yes": group[1],
                "Lift": group[1] / group[0] if group[0] > 0 else np.nan
            }

    result_df = pd.DataFrame(results).T.sort_values("Lift", ascending=False)
    
    print(f"\n=== IMPACT ({label}) ===")
    print(result_df)
    
    return result_df

In [13]:
# Run all 3 analyses
print("High Engagement Analysis: (Threshold: 75th Percentile)")
impact_raw = compute_lift(features_raw, "RAW (Ever Participated)")
impact_early = compute_lift(features_early, "EARLY (First 2 Interactions)")
impact_collapsed = compute_lift(features_collapsed, "COLLAPSED (Ever in Sequence)")

High Engagement Analysis: (Threshold: 75th Percentile)

=== IMPACT (RAW (Ever Participated)) ===
                              No       Yes      Lift
Counseling Appointment  0.154874  0.744848  4.809373
Funding                 0.196129  0.614583  3.133566
Course                  0.165520  0.473655  2.861625
Event                   0.125658  0.337422  2.685229

=== IMPACT (EARLY (First 2 Interactions)) ===
                              No       Yes      Lift
Counseling Appointment  0.218750  0.632249  2.890281
Course                  0.241133  0.365541  1.515931
Funding                 0.269808  0.349736  1.296243
Event                   0.241514  0.296700  1.228498

=== IMPACT (COLLAPSED (Ever in Sequence)) ===
                              No       Yes      Lift
Counseling Appointment  0.154874  0.744848  4.809373
Funding                 0.196129  0.614583  3.133566
Course                  0.165520  0.473655  2.861625
Event                   0.125658  0.337422  2.685229
